# MXFP4 数值实验 1/2：FP4 网格与 UE8M0 Scale Policy

这个 notebook 只做两件事：

1. **实验 1：FP4 E2M1 本体误差**  
   先不考虑 scale，只研究 `Q_FP4(z)` 这个阶梯量化函数本身。

2. **实验 2：MXFP4 scale 如何 round**  
   固定 FP4 值格式，比较 scale 的 `round down`、`round up`、`nearest`、`用 block MSE 选择 up/down`，以及 compressed-tensors 当前实现的 `repo` scale 语义。

核心问题是：MXFP4 的误差并不是简单地“多一个 scale 误差项”。scale 决定了输入落在 FP4 网格的哪个位置，从而决定是否进入 normal region、subnormal/zero region 或 clipping region。

## 0. 先把问题拆开

MXFP4 的量化可以写成：

$$
\hat{x} = \hat{s}\, Q_{FP4}(x / \hat{s})
$$

其中 `Q_FP4` 是 FP4 E2M1 的值量化，`\hat{s}` 是 UE8M0 power-of-two scale。

为了避免把两个现象混在一起，我们先单独看 FP4 网格，再看 scale policy。

几个重要变量：

$$
s^* = \frac{\operatorname{amax}(b)}{6}, \quad
\alpha = \frac{\hat{s}}{s^*}, \quad
r = \frac{|x|}{\operatorname{amax}(b)}, \quad
z = \frac{|x|}{\hat{s}} = \frac{6r}{\alpha}
$$

直觉：

- `alpha < 1`：scale 被 round down，`z` 被放大，主要风险是 **clipping**。
- `alpha > 1`：scale 被 round up，`z` 被压小，主要风险是 **subnormal / zero**。
- scale 是否好，不只看 `alpha` 离 1 多近，还要看整个 block 里的 `r` 分布。

In [6]:
import math
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

# Make imports work whether the notebook is executed from repo root or opened interactively.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUT_DIR = REPO_ROOT / "experiments" / "mxfp4_experiments_1_2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep everything CPU/deterministic for a numerical study.
torch.manual_seed(0)
np.random.seed(0)
torch.set_printoptions(precision=6, sci_mode=False)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True})

print(f"Repo root: {REPO_ROOT}")
print(f"Outputs:   {OUT_DIR}")

Repo root: /root/repos/compressed-tensors
Outputs:   /root/repos/compressed-tensors/experiments/mxfp4_experiments_1_2_outputs


## 1. FP4 E2M1 的精确网格

本实验使用 compressed-tensors 中的 FP4 语义。正值格点是：

$$
\{0, 0.5, 1, 1.5, 2, 3, 4, 6\}
$$

rounding midpoint 是：

$$
0.25, 0.75, 1.25, 1.75, 2.5, 3.5, 5.0
$$

注意最后一个 midpoint 是 5.0：只要 scaled value 大于 5，就会被量化成 6。真正的 clipping 发生在 `z > 6`，因为 dequant 后最多只能回到 6 倍 scale。

In [7]:
FP4_GRID = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
FP4_BOUNDS = torch.tensor([0.25, 0.75, 1.25, 1.75, 2.5, 3.5, 5.0])


def fp4_round(x: torch.Tensor) -> torch.Tensor:
    """Vectorized FP4 E2M1 rounding that matches compressed_tensors.quant_args.FP4_E2M1_DATA.cast_to_fp4."""
    sign = torch.sign(x)
    a = torch.abs(x)
    out = torch.zeros_like(a)
    out = torch.where((a > 0.25) & (a < 0.75), torch.tensor(0.5, dtype=a.dtype), out)
    out = torch.where((a >= 0.75) & (a <= 1.25), torch.tensor(1.0, dtype=a.dtype), out)
    out = torch.where((a > 1.25) & (a < 1.75), torch.tensor(1.5, dtype=a.dtype), out)
    out = torch.where((a >= 1.75) & (a <= 2.5), torch.tensor(2.0, dtype=a.dtype), out)
    out = torch.where((a > 2.5) & (a < 3.5), torch.tensor(3.0, dtype=a.dtype), out)
    out = torch.where((a >= 3.5) & (a <= 5.0), torch.tensor(4.0, dtype=a.dtype), out)
    out = torch.where(a > 5.0, torch.tensor(6.0, dtype=a.dtype), out)
    return out * sign


def quantize_dequant_with_scale(x: torch.Tensor, scale: torch.Tensor | float) -> torch.Tensor:
    scale_t = torch.as_tensor(scale, dtype=x.dtype, device=x.device)
    return fp4_round(x / scale_t) * scale_t


def relative_error(x_hat: torch.Tensor, x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return (x_hat - x).abs() / (x.abs() + eps)


# Quick sanity check against the repository implementation on representative points.
from compressed_tensors.quantization.quant_args import FP4_E2M1_DATA

test = torch.tensor([-7.0, -5.2, -4.0, -2.7, -1.3, -0.4, 0.0, 0.2, 0.4, 0.8, 1.3, 2.8, 4.3, 5.2, 7.0])
ours = fp4_round(test)
repo = FP4_E2M1_DATA.cast_to_fp4(test.clone())
print("test: ", test)
print("ours: ", ours)
print("repo: ", repo)
print("match:", torch.equal(ours, repo))

test:  tensor([-7.000000, -5.200000, -4.000000, -2.700000, -1.300000, -0.400000,
         0.000000,  0.200000,  0.400000,  0.800000,  1.300000,  2.800000,
         4.300000,  5.200000,  7.000000])
ours:  tensor([-6.000000, -6.000000, -4.000000, -3.000000, -1.500000, -0.500000,
         0.000000,  0.000000,  0.500000,  1.000000,  1.500000,  3.000000,
         4.000000,  6.000000,  6.000000])
repo:  tensor([-6.000000, -6.000000, -4.000000, -3.000000, -1.500000, -0.500000,
         0.000000,  0.000000,  0.500000,  1.000000,  1.500000,  3.000000,
         4.000000,  6.000000,  6.000000])
match: True


## 1.1 Dense sweep：`z` 从 0 到 8

这里的 `z` 是已经被 scale 除过之后的值，也就是送进 FP4 的输入。

我们关心三个量：

$$
q = Q_{FP4}(z), \quad |q-z|, \quad \frac{|q-z|}{z}
$$

这一步不涉及 MXFP4 的 scale，所以它给出的是 FP4 值格式本身的误差地形。

In [8]:
z = torch.linspace(0.0, 8.0, 20001)
q = fp4_round(z)
abs_err = (q - z).abs()
rel_err = abs_err / torch.clamp(z, min=1e-12)

regions = {
    "zero/dead zone: 0 < z <= 0.25": (z > 0.0) & (z <= 0.25),
    "subnormal-ish: 0.25 < z < 1": (z > 0.25) & (z < 1.0),
    "normal target: 1 <= z <= 6": (z >= 1.0) & (z <= 6.0),
    "above FP4 max: z > 6": z > 6.0,
}

summary_rows = []
for name, mask in regions.items():
    summary_rows.append((
        name,
        int(mask.sum()),
        float(abs_err[mask].max()),
        float(rel_err[mask].max()),
        float(rel_err[mask].mean()),
    ))

print("FP4 scalar sweep summary")
print("region | count | max_abs_err | max_rel_err | mean_rel_err")
for row in summary_rows:
    print(f"{row[0]:36s} | {row[1]:5d} | {row[2]:11.6f} | {row[3]:11.6f} | {row[4]:12.6f}")

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)

axes[0].plot(z.numpy(), q.numpy(), lw=2)
axes[0].plot(z.numpy(), z.numpy(), "--", color="gray", lw=1, label="identity")
axes[0].set_ylabel("Q_FP4(z)")
axes[0].set_title("FP4 E2M1 quantization staircase")
axes[0].legend()

axes[1].plot(z.numpy(), abs_err.numpy(), color="tab:orange", lw=2)
axes[1].set_ylabel("absolute error")
axes[1].set_title("Absolute error in scaled domain")

axes[2].plot(z.numpy(), rel_err.clamp(max=2.0).numpy(), color="tab:red", lw=2)
axes[2].axhline(0.25, color="black", ls="--", lw=1, label="25%")
axes[2].set_ylabel("relative error, clipped at 200%")
axes[2].set_xlabel("scaled value z")
axes[2].set_title("Relative error: small z and clipped z are the dangerous regions")
axes[2].legend()

for ax in axes:
    for b in FP4_BOUNDS:
        ax.axvline(float(b), color="gray", alpha=0.25, lw=1)
    ax.axvspan(1.0, 6.0, color="green", alpha=0.08, label=None)
    ax.axvline(6.0, color="purple", alpha=0.6, lw=1.5)

plt.tight_layout()
fig_path = OUT_DIR / "experiment1_fp4_grid.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved: {fig_path}")

FP4 scalar sweep summary
region | count | max_abs_err | max_rel_err | mean_rel_err
zero/dead zone: 0 < z <= 0.25        |   625 |    0.250000 |    1.000000 |     1.000000
subnormal-ish: 0.25 < z < 1          |  1874 |    0.250000 |    0.996805 |     0.241893
normal target: 1 <= z <= 6           | 12501 |    1.000000 |    0.200000 |     0.091969
above FP4 max: z > 6                 |  5000 |    2.000000 |    0.250000 |     0.136979
Saved: /root/repos/compressed-tensors/experiments/mxfp4_experiments_1_2_outputs/experiment1_fp4_grid.png


## 1.2 实验 1 的读法

这张图应该这样读：

- `0 < z <= 0.25`：全部变成 0，relative error 可以接近 100%。这就是 small value 最危险的地方。
- `0.25 < z < 1`：虽然不全是 0，但格点非常稀疏，relative error 仍然会很大。
- `1 <= z <= 6`：这是我们希望大多数元素落入的区间。这里 FP4 的误差由 E2M1 网格控制。
- `z > 6`：dequant 后最多只能返回 6 倍 scale，所以这是 clipping 区间。

因此，MXFP4 的 scale 问题可以重新表述为：**scale policy 能否让 block 里的大多数元素落在 `1 <= z <= 6`，并且不要让 absmax 自己跑到 `z > 6`。**

# 2. Scale policy：round down、round up、nearest、MSE select

MXFP4 的 scale 是 power-of-two，因此理想 scale

$$
s^* = \frac{\operatorname{amax}}{6}
$$

通常不能被精确表示。我们比较几种选择：

$$
\hat{s}_{down} = 2^{\lfloor \log_2 s^* \rfloor}
$$

$$
\hat{s}_{up} = 2^{\lceil \log_2 s^* \rceil}
$$

$$
\hat{s}_{nearest} = 2^{\operatorname{round}(\log_2 s^*)}
$$

还有一个更聪明的策略：

$$
\hat{s}_{mse} = \arg\min_{s \in \{\hat{s}_{down}, \hat{s}_{up}\}}
\frac{1}{n}\sum_i\left(s Q_{FP4}(x_i/s) - x_i\right)^2
$$

`nearest` 只看 scale 本身离哪个 power-of-two 近；`mse_select` 会看整个 block 量化后的真实误差。

In [9]:
from compressed_tensors.quantization.utils.mxfp_utils import generate_mx_scales, maybe_convert_from_mx_exp
from compressed_tensors.quantization.quant_args import QuantizationArgs, QuantizationStrategy, QuantizationType

MX_ARGS = QuantizationArgs(
    num_bits=4,
    type=QuantizationType.FLOAT,
    strategy=QuantizationStrategy.GROUP,
    group_size=32,
    symmetric=True,
    scale_dtype=torch.uint8,
    zp_dtype=torch.uint8,
)


def pow2_floor(s_star: torch.Tensor) -> torch.Tensor:
    return torch.pow(2.0, torch.floor(torch.log2(s_star)))


def pow2_ceil(s_star: torch.Tensor) -> torch.Tensor:
    return torch.pow(2.0, torch.ceil(torch.log2(s_star)))


def pow2_nearest(s_star: torch.Tensor) -> torch.Tensor:
    return torch.pow(2.0, torch.round(torch.log2(s_star)))


def repo_mxfp4_scale_from_amax(amax: torch.Tensor) -> torch.Tensor:
    """Return the float scale implied by compressed-tensors MX scale generation."""
    # generate_mx_scales returns biased E8M0 exponents. In the normal qparam path,
    # rounding to uint8 keeps the original floating dtype, then maybe_convert_from_mx_exp
    # turns those exponent values into power-of-two float scales.
    exp_scale = generate_mx_scales(amax.to(torch.float32), num_bits=4)
    exp_scale = torch.round(torch.clamp(exp_scale, 0, 255))
    return maybe_convert_from_mx_exp(MX_ARGS, exp_scale)


def scale_candidates_from_block(block: torch.Tensor) -> dict[str, torch.Tensor]:
    amax = block.abs().max()
    s_star = amax / 6.0
    s_down = pow2_floor(s_star)
    s_up = pow2_ceil(s_star)
    s_nearest = pow2_nearest(s_star)
    s_repo = repo_mxfp4_scale_from_amax(amax.reshape(1)).reshape(())

    x_hat_down = quantize_dequant_with_scale(block, s_down)
    x_hat_up = quantize_dequant_with_scale(block, s_up)
    mse_down = torch.mean((x_hat_down - block) ** 2)
    mse_up = torch.mean((x_hat_up - block) ** 2)
    s_mse = torch.where(mse_down <= mse_up, s_down, s_up)

    return {
        "exact": s_star,
        "down": s_down,
        "up": s_up,
        "nearest": s_nearest,
        "mse_select": s_mse,
        "repo": s_repo,
    }


def block_metrics(block: torch.Tensor, scale: torch.Tensor) -> dict[str, float]:
    amax = block.abs().max()
    s_star = amax / 6.0
    z_abs = block.abs() / scale
    x_hat = quantize_dequant_with_scale(block, scale)
    err = x_hat - block
    rel = relative_error(x_hat, block)
    absmax_z = float(amax / scale)
    return {
        "scale": float(scale),
        "alpha": float(scale / s_star),
        "absmax_z": absmax_z,
        "absmax_clips": float(absmax_z > 6.0),
        "mse": float(torch.mean(err ** 2)),
        "mae": float(torch.mean(err.abs())),
        "rel_p50": float(torch.quantile(rel, 0.50)),
        "rel_p95": float(torch.quantile(rel, 0.95)),
        "zero_frac": float((fp4_round(block / scale) == 0).float().mean()),
        "subnormal_frac_z_lt_1": float((z_abs < 1.0).float().mean()),
        "clip_frac_z_gt_6": float((z_abs > 6.0).float().mean()),
    }


def print_metric_table(rows: list[tuple[str, dict[str, float]]], title: str) -> None:
    print(title)
    headers = ["policy", "alpha", "absmax_z", "clip?", "mse", "mae", "rel_p50", "rel_p95", "zero_frac", "z<1", "z>6"]
    print(" | ".join(headers))
    for name, m in rows:
        print(
            f"{name:10s} | {m['alpha']:7.4f} | {m['absmax_z']:8.4f} | {int(m['absmax_clips']):5d} | "
            f"{m['mse']:9.6g} | {m['mae']:9.6g} | {m['rel_p50']:8.4f} | {m['rel_p95']:8.4f} | "
            f"{m['zero_frac']:9.4f} | {m['subnormal_frac_z_lt_1']:5.3f} | {m['clip_frac_z_gt_6']:5.3f}"
        )

## 2.1 只扫 scale phase：`alpha` 如何变化

这一节只研究一个问题：**理想 scale `s*` 落在两个相邻 power-of-two 之间时，不同 round policy 会造成多大的 scale 偏差。**

MXFP4 的 UE8M0 scale 只能取 power-of-two，例如：

```text
..., 1/8, 1/4, 1/2, 1, 2, 4, 8, ...
```

但理想 scale 是：

$$
s^* = \frac{\operatorname{amax}}{6}
$$

它通常不是 power-of-two。比如 `s* = 1.3` 时，MXFP4 只能在 `1` 和 `2` 之间选一个；`s* = 5.2` 时，只能在 `4` 和 `8` 之间选一个。

关键观察：`1.3` 在 `[1, 2)` 里的相对位置，和 `5.2` 在 `[4, 8)` 里的相对位置，本质上是同一个问题，只是整体乘了一个 `2^k`。所以我们把 `s*` 写成：

$$
s^* = 2^k \cdot 2^\phi, \quad \phi \in [0, 1)
$$

这里：

- `k` 是整数，表示 `s*` 落在哪个 power-of-two 区间。
- `phi` 是小数部分，表示 `s*` 在这个区间里的位置。
- scale rounding 的相对误差只依赖 `phi`，不依赖 `k`。

举例：

- `phi = 0`：`s*` 正好是 power-of-two，所有 policy 都没有 scale 误差。
- `phi = 0.25`：`s*` 比 lower power-of-two 大一点，round down 偏小，round up 偏大。
- `phi = 0.5`：`s*` 正好在 log 空间的中点，nearest 的切换点在这里。
- `phi -> 1`：`s*` 很接近 upper power-of-two，round up 几乎无误差，round down 误差很大。

对 floor / ceil 来说可以直接写出：

$$
\alpha_{down} = \frac{2^k}{2^k 2^\phi} = 2^{-\phi}
$$

$$
\alpha_{up} = \frac{2^{k+1}}{2^k 2^\phi} = 2^{1-\phi}
$$

所以扫 `phi in [0, 1)`，就是把所有可能的 scale 落点都扫一遍。

这一节画三个量：

- `alpha = s_hat / s*`：scale 本身偏大还是偏小。
- `absmax_z = amax / s_hat = 6 / alpha`：block 里最大值被送进 FP4 后落在哪里。
- absmax 元素的原始域相对误差。

这里的 absmax 元素非常重要：如果 `alpha < 1`，则 `absmax_z > 6`，absmax 自己会 clip。

### 2.1.1 这里的 `s_hat` 是什么？

`s_hat` 写作 $\hat{s}$，意思是 **MXFP4 最后实际使用的量化 scale**。

它和理想 scale `s*` 不一样：

$$
s^* = \frac{\operatorname{amax}}{6}
$$

这个 `s*` 是“如果 scale 可以是任意 FP32 数值，我们最想用的 scale”。因为这样 block 里的最大值会刚好被缩放到 FP4 最大值 6：

$$
\frac{\operatorname{amax}}{s^*} = 6
$$

但是 MXFP4 的 UE8M0 scale 只能表示 power-of-two，所以真正能用的是：

$$
\hat{s} \in \{..., 2^{-2}, 2^{-1}, 2^0, 2^1, 2^2, ...\}
$$

因此：

- `s*`：理想 scale，通常不能被 MXFP4 精确表示。
- `s_hat` / $\hat{s}$：实际 scale，必须是某个 power-of-two。
- `alpha = s_hat / s*`：实际 scale 相对理想 scale 的偏差。

如果 `alpha < 1`，说明实际 scale 偏小，所有 scaled values 都会被放大。  
如果 `alpha > 1`，说明实际 scale 偏大，所有 scaled values 都会被压小。

### 2.1.2 `nearest` 和 `repo` 是什么关系？

`nearest` 是一个理论 baseline：直接把理想 scale `s* = amax / 6` round 到最近的 power-of-two。

$$
\hat{s}_{nearest} = 2^{\operatorname{round}(\log_2 s^*)}
$$

`repo` 是 compressed-tensors 当前实现的 MXFP4 scale 生成规则。它不是直接 round `s*`，而是先对 `amax` 做一个偏保守的 power-of-two rounding，然后再减去 FP4 的 exponent offset：

```python
scale_power_2 = round_to_power_2(amax)
s_hat_repo = scale_power_2 / 4
```

这里 `/ 4` 来自 FP4 最大值 6 的 exponent offset：`floor(log2(6)) = 2`。

因此二者关系是：

- `nearest`：问“`s*` 离哪个 power-of-two 最近？”
- `repo`：问“`amax` 应该用哪个 power-of-two bucket，再除以 4 得到 scale？”

因为 FP4 最大值是 6，不是 power-of-two，`repo` 和 `nearest` 的切换边界不会相同。

在 `s* in [1, 2)` 这个区间里：

- `nearest` 在 `s* = sqrt(2) ≈ 1.414` 处从 1 切到 2。
- `repo` 大约在 `amax = 7`，也就是 `s* = 7/6 ≈ 1.167` 处从 1 切到 2。

所以 `repo` 比 `nearest` 更早 round up。结果是：

- `repo` 更少 clip 大值。
- `repo` 更容易把小值压小，增加 subnormal/zero 风险。

这也是上面 summary 里看到的：`repo` 的 `clip_phase_frac = 0.222`，小于 `nearest` 的 `0.500`。

In [10]:
phase = torch.linspace(0.0, 1.0, 2001)[:-1]  # [0, 1)
s_star_phase = torch.pow(2.0, phase)
amax_phase = 6.0 * s_star_phase

scale_by_policy = {
    "down": pow2_floor(s_star_phase),
    "up": pow2_ceil(s_star_phase),
    "nearest": pow2_nearest(s_star_phase),
    "repo": repo_mxfp4_scale_from_amax(amax_phase),
}

phase_stats = {}
for name, scale in scale_by_policy.items():
    alpha = scale / s_star_phase
    absmax_z = amax_phase / scale
    # absmax dequantized value: scale * Q_FP4(absmax / scale)
    absmax_hat = scale * fp4_round(absmax_z)
    absmax_rel = (absmax_hat - amax_phase).abs() / amax_phase
    phase_stats[name] = {"alpha": alpha, "absmax_z": absmax_z, "absmax_rel": absmax_rel}

print("Scale phase summary")
print("policy | alpha_min | alpha_max | absmax_z_min | absmax_z_max | max_absmax_rel | clip_phase_frac")
for name, st in phase_stats.items():
    alpha = st["alpha"]
    absmax_z = st["absmax_z"]
    absmax_rel = st["absmax_rel"]
    print(
        f"{name:7s} | {float(alpha.min()):9.6f} | {float(alpha.max()):9.6f} | "
        f"{float(absmax_z.min()):12.6f} | {float(absmax_z.max()):12.6f} | "
        f"{float(absmax_rel.max()):14.6f} | {float((absmax_z > 6).float().mean()):15.6f}"
    )

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)
colors = {"down": "tab:red", "up": "tab:blue", "nearest": "tab:green", "repo": "tab:purple"}
for name, st in phase_stats.items():
    axes[0].plot(phase.numpy(), st["alpha"].numpy(), label=name, color=colors[name])
    axes[1].plot(phase.numpy(), st["absmax_z"].numpy(), label=name, color=colors[name])
    axes[2].plot(phase.numpy(), st["absmax_rel"].numpy(), label=name, color=colors[name])

axes[0].axhline(1.0, color="black", lw=1, ls="--")
axes[0].set_ylabel("alpha = s_hat / s*")
axes[0].set_title("Scale multiplicative error over log2 phase")

axes[1].axhline(6.0, color="black", lw=1, ls="--", label="FP4 max = 6")
axes[1].set_ylabel("absmax_z = 6 / alpha")
axes[1].set_title("Where the block absmax lands in FP4 scaled domain")

axes[2].set_ylabel("absmax relative error")
axes[2].set_xlabel("fractional phase of log2(s*)")
axes[2].set_title("Actual dequantized error of the absmax element")

for ax in axes:
    ax.legend(ncols=4)
plt.tight_layout()
fig_path = OUT_DIR / "experiment2_scale_phase_alpha.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved: {fig_path}")

Scale phase summary
policy | alpha_min | alpha_max | absmax_z_min | absmax_z_max | max_absmax_rel | clip_phase_frac
down    |  0.500173 |  1.000000 |     6.000000 |    11.995842 |       0.499827 |        0.999500
up      |  1.000000 |  1.999307 |     3.001040 |     6.000000 |       0.199971 |        0.000000
nearest |  0.707107 |  1.413724 |     4.244111 |     8.485281 |       0.292893 |        0.500000
repo    |  0.857376 |  1.714158 |     3.500261 |     6.998096 |       0.199971 |        0.222000


Saved: /root/repos/compressed-tensors/experiments/mxfp4_experiments_1_2_outputs/experiment2_scale_phase_alpha.png


## 2.2 一个容易误解的点：absmax clipping 不是 50% 原始域误差

如果 `alpha < 1`，block absmax 的 scaled value 是：

$$
z_{max} = \frac{6}{\alpha} > 6
$$

它会被 FP4 clamp 成 6。反量化以后：

$$
\hat{x}_{max} = \hat{s} \cdot 6 = \alpha s^* \cdot 6 = \alpha x_{max}
$$

所以 absmax 元素在**原始域**的相对误差是：

$$
\frac{|\hat{x}_{max} - x_{max}|}{x_{max}} = 1 - \alpha
$$

如果 nearest power-of-two 的最坏 `alpha = 2^{-1/2} \approx 0.707`，那么 absmax 原始域误差约为 `29.3%`，不是 50%。

50% 常见于另一个表达：scaled domain 里 `z_max = 8.49` 被 clamp 到 6，scaled-domain 相对误差是 `(8.49 - 6) / 8.49 = 29.3%`，依然不是 50%。所以这个 notebook 会用实际数值把这个点验证出来。

## 2.3 用 block MSE 决定 round up 还是 round down

现在进入真正的 MXFP4 block。一个 block 里有 32 个元素，scale 由 absmax 决定，但 MSE 由所有元素共同决定。

我们构造 4 种 block，全部固定 `amax = 1`：

1. `near_absmax`：大部分元素都接近 absmax。
2. `uniform_r`：元素相对大小 `r` 在 `[0, 1]` 里比较均匀。
3. `outlier_many_small`：一个 outlier，其他元素都很小。
4. `log_uniform_r`：`log r` 均匀，模拟大量跨数量级的小值。

这一步会展示：

- `round down` 保护小值，但可能 clip 大值。
- `round up` 保护大值不 clip，但会把更多小值推向 zero/subnormal。
- `nearest` 不一定等于 MSE 最优。
- `mse_select` 会根据 block 的 `r` 分布改变选择。

In [11]:
def make_block(kind: str, amax: float = 1.0, n: int = 32) -> torch.Tensor:
    if kind == "near_absmax":
        r = torch.linspace(0.72, 1.0, n)
    elif kind == "uniform_r":
        r = torch.linspace(0.02, 1.0, n)
    elif kind == "outlier_many_small":
        r = torch.cat([torch.tensor([1.0]), torch.linspace(0.005, 0.12, n - 1)])
    elif kind == "log_uniform_r":
        r = torch.exp(torch.linspace(math.log(0.003), math.log(1.0), n))
    else:
        raise ValueError(kind)

    # Alternate signs so symmetric behavior is also exercised. amax remains positive in abs value.
    signs = torch.where(torch.arange(n) % 2 == 0, 1.0, -1.0)
    block = amax * r * signs
    block[0] = amax
    return block.to(torch.float32)

block_kinds = ["near_absmax", "uniform_r", "outlier_many_small", "log_uniform_r"]
policies_to_compare = ["exact", "down", "up", "nearest", "mse_select", "repo"]

for kind in block_kinds:
    block = make_block(kind)
    scales = scale_candidates_from_block(block)
    rows = [(policy, block_metrics(block, scales[policy])) for policy in policies_to_compare]
    print()
    print_metric_table(rows, title=f"Block kind: {kind}")
    print(f"MSE-select chose: {'down' if torch.equal(scales['mse_select'], scales['down']) else 'up'}")


Block kind: near_absmax
policy | alpha | absmax_z | clip? | mse | mae | rel_p50 | rel_p95 | zero_frac | z<1 | z>6
exact      |  1.0000 |   6.0000 |     0 | 0.0104489 | 0.0902822 |   0.1143 |   0.1899 |    0.0000 | 0.000 | 0.000
down       |  0.7500 |   8.0000 |     1 | 0.0209798 |  0.120988 |   0.1369 |   0.2462 |    0.0000 | 0.000 | 0.906
up         |  1.5000 |   4.0000 |     0 | 0.00452571 | 0.0551714 |   0.0607 |   0.1327 |    0.0000 | 0.000 | 0.000
nearest    |  0.7500 |   8.0000 |     1 | 0.0209798 |  0.120988 |   0.1369 |   0.2462 |    0.0000 | 0.000 | 0.906
mse_select |  1.5000 |   4.0000 |     0 | 0.00452571 | 0.0551714 |   0.0607 |   0.1327 |    0.0000 | 0.000 | 0.000
repo       |  1.5000 |   4.0000 |     0 | 0.00452571 | 0.0551714 |   0.0607 |   0.1327 |    0.0000 | 0.000 | 0.000
MSE-select chose: up

Block kind: uniform_r
policy | alpha | absmax_z | clip? | mse | mae | rel_p50 | rel_p95 | zero_frac | z<1 | z>6
exact      |  1.0000 |   6.0000 |     0 | 0.00396763 | 0.0471505

### 2.3.1 怎么读 block 表格？

每一行是同一个 block 使用一种 scale policy 之后的量化结果。

列的含义：

- `policy`：scale 怎么选。`exact` 用理想 FP32 scale；`down` 向下取 power-of-two；`up` 向上取 power-of-two；`nearest` 选最近的 power-of-two；`mse_select` 在 down/up 里选 block MSE 更小的；`repo` 是 compressed-tensors 当前实现。
- `alpha`：$\alpha = \hat{s}/s^*$。小于 1 表示 scale 偏小，大于 1 表示 scale 偏大。
- `absmax_z`：block 最大值进入 FP4 前的 scaled value，等于 $\operatorname{amax}/\hat{s}=6/\alpha$。
- `clip?`：absmax 是否超过 FP4 最大值 6。如果 `absmax_z > 6`，就是 1。
- `mse` / `mae`：原始域里的平均平方误差和平均绝对误差。
- `rel_p50` / `rel_p95`：逐元素相对误差的中位数和 95 分位数。
- `zero_frac`：有多少元素被 FP4 round 成 0。
- `z<1`：有多少元素落在 FP4 normal range 以下。
- `z>6`：有多少元素超过 FP4 最大值，发生 clipping 风险。

读表时不要只看 `alpha`。真正要看的是：这个 `alpha` 把 block 里的元素推到了 FP4 网格的哪些区域，以及最终 `mse` 是否更小。

## 2.4 MSE-select 的相图：什么时候选 down，什么时候选 up

上面只看了几个手工 block。现在系统地扫两个变量：

- `phase = frac(log2(s*))`：决定 floor/up 两个 scale 离 ideal scale 有多远。
- `small_r`：除 absmax 外，其余元素的相对大小。

构造 block：

$$
[1, r, r, r, \dots, r]
$$

如果 `r` 很小，round up 会把小值更容易压到 0，MSE-select 可能偏向 down。  
如果 `r` 接近 1，round down 会 clip 很多大值，MSE-select 可能偏向 up。

这个图是理解 MXFP4 scale policy 的关键：**scale rounding 的最优方向依赖 block 内分布，不只依赖 absmax。**

In [12]:
phase_grid = torch.linspace(0.001, 0.999, 220)
r_grid = torch.linspace(0.001, 1.0, 220)
choose_up = torch.zeros((len(r_grid), len(phase_grid)))
mse_ratio_up_over_down = torch.zeros_like(choose_up)

for i, r in enumerate(r_grid):
    base_r = torch.cat([torch.tensor([1.0]), torch.full((31,), float(r))])
    signs = torch.where(torch.arange(32) % 2 == 0, 1.0, -1.0)
    base_r = base_r * signs
    base_r[0] = 1.0

    for j, ph in enumerate(phase_grid):
        # Choose amax so that s* has desired log2 phase. Since s*=amax/6, amax=6*2^phase.
        amax = 6.0 * torch.pow(torch.tensor(2.0), ph)
        block = base_r * amax
        s_star = amax / 6.0
        s_down = pow2_floor(s_star)
        s_up = pow2_ceil(s_star)
        x_hat_down = quantize_dequant_with_scale(block, s_down)
        x_hat_up = quantize_dequant_with_scale(block, s_up)
        mse_down = torch.mean((x_hat_down - block) ** 2)
        mse_up = torch.mean((x_hat_up - block) ** 2)
        choose_up[i, j] = float(mse_up < mse_down)
        mse_ratio_up_over_down[i, j] = float(mse_up / (mse_down + 1e-30))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

im0 = axes[0].imshow(
    choose_up.numpy(),
    origin="lower",
    aspect="auto",
    extent=[float(phase_grid.min()), float(phase_grid.max()), float(r_grid.min()), float(r_grid.max())],
    cmap="coolwarm",
    vmin=0,
    vmax=1,
)
axes[0].set_title("MSE-selected policy: blue=down, red=up")
axes[0].set_xlabel("fractional phase of log2(s*)")
axes[0].set_ylabel("small element ratio r")

log_ratio = torch.log10(mse_ratio_up_over_down.clamp(min=1e-6, max=1e6))
im1 = axes[1].imshow(
    log_ratio.numpy(),
    origin="lower",
    aspect="auto",
    extent=[float(phase_grid.min()), float(phase_grid.max()), float(r_grid.min()), float(r_grid.max())],
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
)
axes[1].set_title("log10(MSE_up / MSE_down)")
axes[1].set_xlabel("fractional phase of log2(s*)")
fig.colorbar(im1, ax=axes[1], label="negative: up better, positive: down better")

plt.tight_layout()
fig_path = OUT_DIR / "experiment2_mse_select_phase_map.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print(f"Saved: {fig_path}")

print(f"Overall fraction selecting up: {choose_up.mean().item():.3f}")

Saved: /root/repos/compressed-tensors/experiments/mxfp4_experiments_1_2_outputs/experiment2_mse_select_phase_map.png
Overall fraction selecting up: 0.753


## 2.5 对实验 2 的预期理解

看完这些图和表，应该形成下面的判断：

1. `round down` 不是“更精确”，它只是让 scale 偏小。结果是所有 scaled values 变大，absmax 更容易 clip。
2. `round up` 不是“更保守”，它只是让 scale 偏大。结果是所有 scaled values 变小，小元素更容易进入 zero/subnormal。
3. `nearest` 优化的是 scale 自己的相对误差，不是 block 量化 MSE。
4. `mse_select` 才是在问真正的问题：这个 block 用 up 还是 down，哪一个 dequant 后更接近原值？
5. 对 outlier-heavy block，很多小值可能让 up 的 zero/subnormal 代价变大；对很多元素都接近 absmax 的 block，down 的 clipping 代价会变大。

所以 MXFP4 的 scale policy 本质上是在 trade off：**保护大值不 clip** vs **保护小值不被压成 zero/subnormal**。

## 3. 小结

实验 1 给出了 FP4 的底层事实：FP4 不是在所有数值区间都一样好。它希望 scaled values 尽量落在 `1 <= z <= 6`。

实验 2 说明了 MXFP4 的 scale 为什么关键：UE8M0 只能选择 power-of-two scale，选择 down/up 会系统性地把整个 block 的 `z` 放大或压小。

最重要的结论是：

$$
\text{best scale direction depends on the block distribution, not only on amax.}
$$

因此，如果只看 `s*` 离哪个 power-of-two 更近，可能错过真正的 MSE 最优选择。下一步如果继续扩展，可以把实验 2 的 MSE-select 接到真实 activation / weight block 上，统计真实模型里 floor/up/nearest/repo 各自的选择频率和误差差异。